# VGG16 Feature Extractor + SVM — Eye ROI Deepfake Experiment
## SSOT v2.0 Uyumlu Sürüm

Bu notebook, **Deepfake Detection — Ekip Standardı v2.0 (SSOT)** kurallarına göre
Kader'in `Deney 1 / Göz / eye_roi_output` veri yapısı için revize edilmiştir.

### Model akışı

`Combined Eye ROI → VGG16 (ImageNet, include_top=False, GlobalAveragePooling) → 512-D feature → StandardScaler → SVM → REAL / FAKE`

### SSOT uyarlamaları

- Ham / mevcut ROI verisi **salt okunur** kabul edilir; hiçbir kaynak dosya silinmez, taşınmaz veya üzerine yazılmaz.
- Her çalışma benzersiz bir **RUN_ID** alır:
  `YYYYMMDD_HHMM_eye_vgg16_svm_seed42`
- Çıktılar yalnızca `Sonuçlar/outputs/<RUN_ID>/` altına yazılır.
- `config_resolved.yaml`, environment, requirements, manifest, audit, metrics ve prediction çıktıları saklanır.
- Metadata şema testi ve veri muhasebesi yapılır.
- Split güvenliği `source_video` üzerinden doğrulanabilir ise **zorunlu olarak** kontrol edilir.
- Exact-content SHA-256 cross-split leakage kontrolü her durumda yapılır.
- StandardScaler yalnızca **train** üzerinde öğrenilir; final scaler ise yalnızca **train+val** üzerinde fit edilir.
- Test seti hiperparametre / threshold / model seçimi için kullanılmaz.
- Atomik kayıt mekanizması ile model/scaler/JSON/YAML/CSV dosyalarının yarım yazılması engellenir.
- Quality gate'ler başarısız olursa deney başlatılmaz veya anında durdurulur.
- Grafikler %100 İngilizce, okunabilir ve kısa kenarı en az 600 px olacak biçimde kaydedilir.

> Bu deneyde VGG16 **eğitilmediği** ve yalnızca sabit feature extractor olduğu için,
> SSOT'deki "2 batch forward/backward smoke test" kuralının backward bölümü bu mimariye
> uygulanamaz. Bunun yerine 2 batch VGG16 forward + feature boyutu/sayısal kontrol +
> küçük SVM fit/predict smoke testi uygulanır ve bu uyarlama audit dosyasına açıkça yazılır.


In [ ]:
# ============================================================
# 0) ENVIRONMENT / DEPENDENCIES
# ============================================================

!pip -q install "scikit-learn>=1.4,<2" "PyYAML>=6,<7" joblib tqdm

import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"
os.environ["PYTHONHASHSEED"] = "42"

import sys
import json
import time
import random
import hashlib
import logging
import platform
import subprocess
import re
import unicodedata
import tempfile
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yaml
import joblib
from tqdm.auto import tqdm
from PIL import Image, UnidentifiedImageError

import tensorflow as tf
from tensorflow.keras.applications import VGG16
from tensorflow.keras.applications.vgg16 import preprocess_input

from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report,
    roc_curve,
    precision_recall_curve,
    average_precision_score,
)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("Python      :", sys.version.split()[0])
print("TensorFlow :", tf.__version__)
print("GPU         :", tf.config.list_physical_devices("GPU"))
print("Seed        :", SEED)


## Yerel kaynak hazırlığı\n\nBu arşivlenmiş proje paketinde dış kaynak bağlantısı kullanılmaz. Gerekli Deney 1 dosyaları repo içindeki yerel kaynak çıktıları, manifestler ve açıklama dosyalarıyla temsil edilir.

### Metadata SSOT Adaptation

Kaynak `metadata.csv` dosyasına dokunulmaz. Notebook, kaynak metadata'yı okuyup
SSOT v2.0 için gerekli alanları **run içine yeni bir canonical manifest** olarak üretir.

Kaynakta bulunmayan bir bilgi **uydurulmaz**. Örneğin gerçek `source_video` bilgisi
yoksa bu alan boş bırakılır ve video-ID split quality gate'i `NOT_VERIFIABLE` olarak raporlanır.


### Quality Gates

Tam feature extraction / SVM eğitimi başlamadan önce:
1. metadata şema testi,
2. veri muhasebesi,
3. benzersiz sample_id testi,
4. train/val/test sınıf testi,
5. exact-content SHA-256 split izolasyonu,
6. source_video mevcutsa video split izolasyonu,
7. 2-batch VGG16 forward + mini-SVM smoke testi

zorunlu olarak çalıştırılır.


In [ ]:
# ============================================================
# 4) SOURCE METADATA → CANONICAL SSOT MANIFEST
# ============================================================

source_metadata = pd.read_csv(METADATA_CSV)

SOURCE_REQUIRED = {"label", "split", "status", "combined_eye_path"}
missing = SOURCE_REQUIRED.difference(source_metadata.columns)
if missing:
    raise ValueError(
        f"Schema gate failed. Missing source columns: {sorted(missing)}"
    )

metadata = source_metadata.copy()

for col in ["label", "split", "status"]:
    metadata[col] = metadata[col].astype(str).str.strip()

metadata["label"] = metadata["label"].str.lower()
metadata["split"] = metadata["split"].str.lower().replace(
    {"validation": "val", "valid": "val"}
)
metadata["status_normalized"] = metadata["status"].str.lower()

allowed_labels = set(CONFIG["dataset"]["labels"])
allowed_splits = set(CONFIG["dataset"]["splits"])
success_values = set(CONFIG["dataset"]["success_values"])

invalid_labels = sorted(
    set(metadata["label"].dropna()) - allowed_labels
)
invalid_splits = sorted(
    set(metadata["split"].dropna()) - allowed_splits
)

if invalid_labels:
    raise ValueError(
        f"Schema gate failed. Invalid labels: {invalid_labels}"
    )

if invalid_splits:
    raise ValueError(
        f"Schema gate failed. Invalid splits: {invalid_splits}"
    )


def first_existing_column(df, names):
    for name in names:
        if name in df.columns:
            return name
    return None


def resolve_roi_path(raw_path, label, split):
    raw = "" if pd.isna(raw_path) else str(raw_path).strip()
    if not raw:
        return None

    p = Path(raw)
    candidates = []

    if p.is_absolute():
        candidates.append(p)

    candidates.append(EYE_ROI_ROOT / raw)
    candidates.append(DENEY1_ROOT / raw)

    p_norm_parts = [norm_name(x) for x in p.parts]
    if "eye_roi_output" in p_norm_parts:
        idx = p_norm_parts.index("eye_roi_output")
        suffix = p.parts[idx + 1:]
        if suffix:
            candidates.append(EYE_ROI_ROOT.joinpath(*suffix))

    # Canonical combined-eye fallback.
    candidates.append(
        EYE_ROI_ROOT / label / split / "combined" / p.name
    )

    seen = set()
    for candidate in candidates:
        key = str(candidate)
        if key in seen:
            continue
        seen.add(key)

        if candidate.is_file():
            return candidate

    return None


source_video_col = first_existing_column(
    metadata,
    [
        "source_video",
        "video_path",
        "video_name",
        "source_video_path",
        "video_id",
    ],
)

frame_index_col = first_existing_column(
    metadata,
    [
        "frame_index",
        "frame_number",
        "frame_idx",
    ],
)

face_index_col = first_existing_column(
    metadata,
    [
        "face_index",
        "face_id",
    ],
)

roi_state_col = first_existing_column(
    metadata,
    [
        "roi_state",
        "eye_state",
    ],
)

sample_id_col = first_existing_column(
    metadata,
    ["sample_id"],
)

skip_reason_col = first_existing_column(
    metadata,
    [
        "skip_reason",
        "reason",
    ],
)

frame_stem_col = first_existing_column(
    metadata,
    ["frame_stem"],
)

source_frame_col = first_existing_column(
    metadata,
    [
        "source_frame",
        "relative_frame_path",
    ],
)


canonical_rows = []

for idx, row in metadata.iterrows():
    label = str(row["label"]).strip().lower()
    split = str(row["split"]).strip().lower()
    original_status = str(row["status"]).strip()
    normalized_status = (
        str(row["status_normalized"]).strip().lower()
    )

    resolved = None
    if normalized_status in success_values:
        resolved = resolve_roi_path(
            row["combined_eye_path"],
            label,
            split,
        )

    source_video = ""
    if source_video_col:
        value = row[source_video_col]
        if pd.notna(value):
            source_video = str(value).strip()

    frame_index = ""
    if frame_index_col:
        value = row[frame_index_col]
        if pd.notna(value):
            frame_index = str(value).strip()

    frame_stem = ""
    if frame_stem_col:
        value = row[frame_stem_col]
        if pd.notna(value):
            frame_stem = str(value).strip()

    source_frame = ""
    if source_frame_col:
        value = row[source_frame_col]
        if pd.notna(value):
            source_frame = str(value).strip()

    face_index = 0
    if face_index_col:
        value = row[face_index_col]
        if pd.notna(value):
            try:
                face_index = int(value)
            except Exception as exc:
                raise ValueError(
                    "Schema gate failed: "
                    f"invalid face_index={value!r} "
                    f"at source row {idx}"
                ) from exc

    roi_state = ""
    if roi_state_col:
        value = row[roi_state_col]
        if pd.notna(value):
            roi_state = str(value).strip()

    skip_reason = ""
    if skip_reason_col:
        value = row[skip_reason_col]
        if pd.notna(value):
            skip_reason = str(value).strip()

    # --------------------------------------------------------
    # IMPORTANT:
    # Source metadata sample_id does NOT have to be unique at
    # ROI-row level. For example, multiple faces/ROIs can share
    # the same source sample/frame identifier.
    #
    # Therefore:
    #   1) preserve source sample_id separately
    #   2) create a NEW canonical ROI-level sample_id from a
    #      deterministic composite key
    #
    # We do NOT append row index just to force uniqueness;
    # exact duplicate ROI records must remain detectable.
    # --------------------------------------------------------
    source_sample_id = ""
    if (
        sample_id_col
        and pd.notna(row[sample_id_col])
        and str(row[sample_id_col]).strip()
    ):
        source_sample_id = str(row[sample_id_col]).strip()

    raw_combined_path = (
        ""
        if pd.isna(row["combined_eye_path"])
        else str(row["combined_eye_path"]).strip()
    )

    resolved_key = str(resolved) if resolved else raw_combined_path

    stable_payload = "|".join(
        [
            source_sample_id,
            source_video,
            frame_index,
            frame_stem,
            source_frame,
            str(face_index),
            label,
            split,
            resolved_key,
        ]
    )

    sample_id = hashlib.sha256(
        stable_payload.encode("utf-8")
    ).hexdigest()[:20]

    status = (
        "SUCCESS"
        if normalized_status in success_values
        else (
            "ERROR"
            if normalized_status in {"error", "failed", "fail"}
            else "SKIPPED"
        )
    )

    canonical_rows.append(
        {
            "sample_id": sample_id,
            "source_sample_id": source_sample_id,
            "source_video": source_video,
            "frame_index": frame_index,
            "face_index": face_index,
            "roi_state": roi_state,
            "label": label,
            "split": split,
            "status": status,
            "skip_reason": skip_reason,
            "sha256": "",
            "output_path": str(resolved) if resolved else "",
            "run_id": RUN_ID,
            "source_row_index": int(idx),
            "source_status": original_status,
            "frame_stem": frame_stem,
            "source_frame": source_frame,
        }
    )


canonical = pd.DataFrame(canonical_rows)

SSOT_COLUMNS = [
    "sample_id",
    "source_sample_id",
    "source_video",
    "frame_index",
    "face_index",
    "roi_state",
    "label",
    "split",
    "status",
    "skip_reason",
    "sha256",
    "output_path",
    "run_id",
]

missing_ssot = set(SSOT_COLUMNS).difference(
    canonical.columns
)
if missing_ssot:
    raise RuntimeError(
        "Canonical schema generation failed: "
        f"{sorted(missing_ssot)}"
    )

# ------------------------------------------------------------
# DATA ACCOUNTING
# ------------------------------------------------------------

total_inputs = len(canonical)
success_count = int(
    (canonical["status"] == "SUCCESS").sum()
)
skipped_count = int(
    (canonical["status"] == "SKIPPED").sum()
)
error_count = int(
    (canonical["status"] == "ERROR").sum()
)

assert (
    total_inputs
    == success_count + skipped_count + error_count
), "Accounting equality failed."


# ------------------------------------------------------------
# DUPLICATE AUDIT
# ------------------------------------------------------------
#
# A duplicated SOURCE sample_id is NOT automatically an error.
# It may legitimately identify a frame while multiple faces /
# eye ROIs exist under that frame.
#
# A duplicated CANONICAL sample_id, however, means the exact
# composite ROI identity occurred more than once and must be
# investigated.
# ------------------------------------------------------------

source_sample_duplicates = canonical[
    canonical["source_sample_id"].ne("")
    & canonical["source_sample_id"].duplicated(
        keep=False
    )
].copy()

if len(source_sample_duplicates):
    atomic_dataframe_csv(
        source_sample_duplicates,
        ARTIFACTS_DIR
        / "source_sample_id_duplicates_audit.csv",
        index=False,
    )

canonical_duplicates = canonical[
    canonical["sample_id"].duplicated(keep=False)
].copy()

if len(canonical_duplicates):
    atomic_dataframe_csv(
        canonical_duplicates,
        ARTIFACTS_DIR
        / "canonical_sample_id_duplicates.csv",
        index=False,
    )

    # If duplicate canonical rows point to the exact same
    # resolved output path, these are true duplicate ROI records.
    duplicate_success = canonical_duplicates[
        canonical_duplicates["status"].eq("SUCCESS")
    ].copy()

    if len(duplicate_success):
        raise RuntimeError(
            "Canonical ROI duplicate records detected. "
            f"Count={len(duplicate_success)}. "
            "See artifacts/canonical_sample_id_duplicates.csv"
        )

    # For SKIPPED/ERROR records an identical source event can
    # appear repeatedly in old metadata. They are not model
    # inputs, so keep only one canonical audit record while
    # preserving the full duplicate report above.
    canonical = canonical.drop_duplicates(
        subset=["sample_id"],
        keep="first",
    ).reset_index(drop=True)

assert canonical["sample_id"].is_unique, (
    "Canonical sample_id generation is not unique."
)


# ------------------------------------------------------------
# SUCCESS PATH VALIDATION
# ------------------------------------------------------------

success_missing_path = canonical[
    canonical["status"].eq("SUCCESS")
    & canonical["output_path"].eq("")
]

if len(success_missing_path):
    atomic_dataframe_csv(
        success_missing_path,
        ARTIFACTS_DIR
        / "missing_success_output_paths.csv",
        index=False,
    )
    raise FileNotFoundError(
        f"{len(success_missing_path)} SUCCESS records "
        "do not resolve to a combined-eye image."
    )


# ------------------------------------------------------------
# MODEL INVENTORY
# ------------------------------------------------------------

inventory = (
    canonical[
        canonical["status"].eq("SUCCESS")
    ]
    .copy()
    .reset_index(drop=True)
)

inventory["class_name"] = inventory["label"]
inventory["numeric_label"] = (
    inventory["label"]
    .map(CONFIG["dataset"]["labels"])
    .astype(np.int64)
)

# Same physical combined-eye file must not be used twice.
duplicate_output_paths = inventory[
    inventory["output_path"].duplicated(keep=False)
].copy()

if len(duplicate_output_paths):
    atomic_dataframe_csv(
        duplicate_output_paths,
        ARTIFACTS_DIR
        / "duplicate_success_output_paths.csv",
        index=False,
    )
    raise RuntimeError(
        "The same SUCCESS combined-eye image is referenced "
        "more than once in metadata. "
        "See artifacts/duplicate_success_output_paths.csv"
    )

for split in CONFIG["dataset"]["splits"]:
    subset = inventory[
        inventory["split"] == split
    ]

    if subset.empty:
        raise RuntimeError(
            f"Quality gate failed: split '{split}' is empty."
        )

    if CONFIG["quality_gates"][
        "require_two_classes_per_split"
    ]:
        if set(
            subset["numeric_label"].unique()
        ) != {0, 1}:
            raise RuntimeError(
                "Quality gate failed: "
                f"split '{split}' does not contain "
                "both REAL and FAKE."
            )


atomic_dataframe_csv(
    canonical[
        SSOT_COLUMNS
        + [
            "source_row_index",
            "source_status",
            "frame_stem",
            "source_frame",
        ]
    ],
    ARTIFACTS_DIR
    / "metadata_canonical_ssot.csv",
    index=False,
)

summary = (
    inventory
    .groupby(["split", "label"])
    .size()
    .unstack(fill_value=0)
    .reindex(
        ["train", "val", "test"],
        fill_value=0,
    )
)

display(summary)

print(
    {
        "source_metadata_rows": total_inputs,
        "canonical_rows_after_nonmodel_dedup": len(canonical),
        "success": int(
            (canonical["status"] == "SUCCESS").sum()
        ),
        "skipped": int(
            (canonical["status"] == "SKIPPED").sum()
        ),
        "error": int(
            (canonical["status"] == "ERROR").sum()
        ),
        "model_inputs": len(inventory),
        "duplicated_source_sample_id_rows": len(
            source_sample_duplicates
        ),
    }
)

logger.info(
    "Canonical manifest ready. "
    "source_rows=%d canonical_rows=%d model_inputs=%d "
    "duplicated_source_sample_id_rows=%d",
    total_inputs,
    len(canonical),
    len(inventory),
    len(source_sample_duplicates),
)


In [ ]:
# ============================================================
# 5) HASHING + SPLIT ISOLATION QUALITY GATES
#    source package-safe + resumable version
# ============================================================

import time
from pathlib import Path

HASH_PROGRESS_FILE = ARTIFACTS_DIR / "sha256_progress.csv"
MISSING_HASH_FILE = ARTIFACTS_DIR / "missing_or_unreadable_images.csv"

HASH_RETRIES = 5
HASH_RETRY_WAIT_SECONDS = 2.0
HASH_SAVE_EVERY = 50


def sha256_file(path: Path, chunk_size=1024 * 1024):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while True:
            chunk = f.read(chunk_size)
            if not chunk:
                break
            h.update(chunk)
    return h.hexdigest()


def normalize_current_eye_path(raw_path):
    raw = "" if pd.isna(raw_path) else str(raw_path).strip()
    if not raw:
        return None

    p = Path(raw)
    try:
        if p.is_file():
            return p
    except OSError:
        pass

    parts_norm = [norm_name(x) for x in p.parts]
    if "eye_roi_output" in parts_norm:
        idx = parts_norm.index("eye_roi_output")
        suffix = p.parts[idx + 1:]
        if suffix:
            rebuilt = EYE_ROI_ROOT.joinpath(*suffix)
            try:
                if rebuilt.is_file():
                    return rebuilt
            except OSError:
                pass
    return None


def resolve_missing_eye_image(row):
    raw_path = str(row.output_path).strip()
    p = Path(raw_path)

    try:
        if p.is_file():
            return p
    except OSError:
        pass

    rebuilt = normalize_current_eye_path(raw_path)
    if rebuilt is not None:
        return rebuilt

    filename = p.name
    label = str(row.label).strip().lower()
    split = str(row.split).strip().lower()
    combined_root = EYE_ROI_ROOT / label / split / "combined"

    candidates = [
        combined_root / filename,
        combined_root / f"{label}_{split}" / filename,
    ]

    for candidate in candidates:
        try:
            if candidate.is_file():
                return candidate
        except OSError:
            pass

    if combined_root.is_dir():
        try:
            matches = list(combined_root.glob(f"*/{filename}"))
            matches += list(combined_root.glob(filename))
        except OSError:
            matches = []

        unique_matches = []
        seen = set()
        for match in matches:
            key = str(match)
            if key in seen:
                continue
            seen.add(key)
            try:
                if match.is_file():
                    unique_matches.append(match)
            except OSError:
                continue

        if len(unique_matches) == 1:
            return unique_matches[0]
        if len(unique_matches) > 1:
            raise RuntimeError(
                "Ambiguous image resolution. More than one file has the same "
                f"name inside {combined_root}: {filename}\n"
                + "\n".join(map(str, unique_matches[:10]))
            )

    return None


def hash_with_retries(path: Path):
    last_error = None
    for attempt in range(1, HASH_RETRIES + 1):
        try:
            if not path.is_file():
                raise FileNotFoundError(str(path))
            digest = sha256_file(path)
            if not digest or len(digest) != 64:
                raise RuntimeError(
                    f"Invalid SHA-256 digest produced for {path}: {digest!r}"
                )
            return digest
        except (FileNotFoundError, OSError, PermissionError, RuntimeError) as exc:
            last_error = exc
            if attempt < HASH_RETRIES:
                time.sleep(HASH_RETRY_WAIT_SECONDS * attempt)
    raise RuntimeError(
        f"Unable to hash after {HASH_RETRIES} attempts: {path}\n"
        f"Last error: {last_error!r}"
    )


completed_hashes = {}
if HASH_PROGRESS_FILE.is_file():
    previous = pd.read_csv(HASH_PROGRESS_FILE)
    required_progress_cols = {"sample_id", "output_path", "sha256"}
    if required_progress_cols.issubset(previous.columns):
        previous = previous[
            previous["sha256"].fillna("").astype(str).str.len().eq(64)
        ].copy()
        completed_hashes = dict(
            zip(
                previous["sample_id"].astype(str),
                previous["sha256"].astype(str),
            )
        )
        print(
            f"SHA256 resume: {len(completed_hashes)} previously completed "
            "records will be reused."
        )

hash_rows = []
missing_rows = []

for position, row in enumerate(
    tqdm(inventory.itertuples(index=False), total=len(inventory), desc="SHA256"),
    start=1,
):
    sample_id = str(row.sample_id)

    if sample_id in completed_hashes:
        hash_rows.append({
            "sample_id": sample_id,
            "output_path": str(row.output_path),
            "sha256": completed_hashes[sample_id],
        })
        continue

    resolved_path = resolve_missing_eye_image(row)

    if resolved_path is None:
        missing_rows.append({
            "sample_id": sample_id,
            "label": row.label,
            "split": row.split,
            "output_path": str(row.output_path),
            "reason": "FILE_NOT_FOUND_AFTER_PATH_RESOLUTION",
        })
    else:
        try:
            digest = hash_with_retries(resolved_path)
            hash_rows.append({
                "sample_id": sample_id,
                "output_path": str(resolved_path),
                "sha256": digest,
            })
            inventory.loc[
                inventory["sample_id"].astype(str).eq(sample_id),
                "output_path",
            ] = str(resolved_path)
        except Exception as exc:
            missing_rows.append({
                "sample_id": sample_id,
                "label": row.label,
                "split": row.split,
                "output_path": str(resolved_path),
                "reason": repr(exc),
            })

    if position % HASH_SAVE_EVERY == 0 or position == len(inventory):
        progress_df = pd.DataFrame(
            hash_rows,
            columns=["sample_id", "output_path", "sha256"],
        )
        atomic_dataframe_csv(progress_df, HASH_PROGRESS_FILE, index=False)

if missing_rows:
    missing_df = pd.DataFrame(missing_rows)
    atomic_dataframe_csv(missing_df, MISSING_HASH_FILE, index=False)
    print("\n❌ SHA256 quality gate stopped.")
    print(f"Unresolved SUCCESS images: {len(missing_df)}")
    display(missing_df.head(20))
    raise FileNotFoundError(
        f"{len(missing_df)} SUCCESS image(s) could not be resolved/read after "
        "source package-safe retries. "
        f"See: {MISSING_HASH_FILE}\n"
        "Already completed hashes were saved, so rerunning this cell will "
        "continue from saved progress instead of starting from zero."
    )

hash_df = pd.DataFrame(hash_rows)
if len(hash_df) != len(inventory):
    raise RuntimeError(
        "SHA256 accounting failed: "
        f"hashed={len(hash_df)}, inventory={len(inventory)}"
    )
if hash_df["sample_id"].duplicated().any():
    raise RuntimeError(
        "SHA256 progress contains duplicate canonical sample_id values."
    )

hash_map = hash_df.set_index("sample_id")["sha256"].to_dict()
inventory["sha256"] = inventory["sample_id"].astype(str).map(hash_map)
if inventory["sha256"].isna().any():
    raise RuntimeError("SHA256 mapping failed for one or more inventory rows.")

canonical["sha256"] = canonical["sample_id"].astype(str).map(hash_map).fillna("")

path_map = inventory.set_index("sample_id")["output_path"].astype(str).to_dict()
success_mask = canonical["status"].eq("SUCCESS")
canonical.loc[success_mask, "output_path"] = (
    canonical.loc[success_mask, "sample_id"]
    .astype(str)
    .map(path_map)
    .fillna(canonical.loc[success_mask, "output_path"])
)

canonical_export_cols = [
    col
    for col in (
        SSOT_COLUMNS
        + ["source_row_index", "source_status", "frame_stem", "source_frame"]
    )
    if col in canonical.columns
]

atomic_dataframe_csv(
    canonical[canonical_export_cols],
    ARTIFACTS_DIR / "metadata_canonical_ssot.csv",
    index=False,
)

split_count_by_hash = (
    inventory.groupby("sha256")["split"]
    .nunique()
    .reset_index(name="split_count")
)
cross_split_hashes = split_count_by_hash[
    split_count_by_hash["split_count"] > 1
]

if len(cross_split_hashes):
    bad_hashes = set(cross_split_hashes["sha256"])
    bad = inventory[inventory["sha256"].isin(bad_hashes)].copy()
    atomic_dataframe_csv(
        bad,
        ARTIFACTS_DIR / "cross_split_content_leakage.csv",
        index=False,
    )
    raise RuntimeError(
        "Split quality gate failed: exact same image content exists in multiple splits."
    )


def valid_string_series(series):
    s = series.fillna("").astype(str).str.strip()
    invalid = {"", "nan", "none", "null", "unknown", "n/a", "na"}
    return ~s.str.casefold().isin(invalid)

source_video_valid_mask = valid_string_series(inventory["source_video"])
source_video_complete = bool(source_video_valid_mask.all())
source_video_gate_status = "NOT_VERIFIABLE"

if source_video_complete and inventory["source_video"].nunique() > 1:
    source_video_gate_status = "PASSED"
    train_ids = set(inventory.loc[inventory["split"] == "train", "source_video"])
    val_ids = set(inventory.loc[inventory["split"] == "val", "source_video"])
    test_ids = set(inventory.loc[inventory["split"] == "test", "source_video"])

    assert train_ids.isdisjoint(val_ids), "Train/Val source_video leakage detected."
    assert train_ids.isdisjoint(test_ids), "Train/Test source_video leakage detected."
    assert val_ids.isdisjoint(test_ids), "Val/Test source_video leakage detected."

audit = {
    "run_id": RUN_ID,
    "schema_gate": "PASSED",
    "accounting_equality": "PASSED",
    "unique_sample_id": "PASSED",
    "sha256_hashed_images": int(len(inventory)),
    "sha256_resume_enabled": True,
    "exact_content_split_isolation": "PASSED",
    "source_video_split_isolation": source_video_gate_status,
    "source_video_complete": source_video_complete,
    "note": (
        "SHA-256 hashing is resumable and source package-safe. "
        "If source_video is NOT_VERIFIABLE, no source-video-based claim is made "
        "and no identifier is fabricated."
    ),
}

atomic_json_dump(audit, ARTIFACTS_DIR / "quality_gate_audit.json")
print("\n✅ SHA256 + split isolation quality gates PASSED")
print(json.dumps(audit, indent=2, ensure_ascii=False))


In [ ]:
# ============================================================
# 6) FEATURE EXTRACTOR BUILD
# ============================================================

img_cfg = CONFIG["image"]
fx_cfg = CONFIG["feature_extractor"]

feature_extractor = VGG16(
    weights=fx_cfg["weights"],
    include_top=fx_cfg["include_top"],
    input_shape=(
        int(img_cfg["height"]),
        int(img_cfg["width"]),
        int(img_cfg["channels"]),
    ),
    pooling=fx_cfg["pooling"],
)
feature_extractor.trainable = bool(fx_cfg["trainable"])

actual_dim = int(feature_extractor.output_shape[-1])
expected_dim = int(fx_cfg["feature_dim"])
if actual_dim != expected_dim:
    raise RuntimeError(
        f"Feature extractor dimension mismatch: expected={expected_dim}, actual={actual_dim}"
    )

atomic_json_dump(
    {
        "architecture": fx_cfg["architecture"],
        "weights": fx_cfg["weights"],
        "include_top": fx_cfg["include_top"],
        "pooling": fx_cfg["pooling"],
        "trainable": fx_cfg["trainable"],
        "input_shape": [
            img_cfg["height"],
            img_cfg["width"],
            img_cfg["channels"],
        ],
        "feature_dim": actual_dim,
    },
    ARTIFACTS_DIR / "feature_extractor.json",
)

print("Feature extractor ready:", feature_extractor.output_shape)


In [ ]:
# ============================================================
# 7) IMAGE LOADING + 2-BATCH SMOKE TEST + FEATURE EXTRACTION
# ============================================================

IMG_SIZE = (int(CONFIG["image"]["width"]), int(CONFIG["image"]["height"]))
BATCH_SIZE = int(CONFIG["image"]["batch_size"])
FEATURE_DIM = int(CONFIG["feature_extractor"]["feature_dim"])

def load_batch(paths):
    arrays = []
    valid_paths = []
    errors = []

    for p in paths:
        try:
            with Image.open(p) as img:
                img = img.convert("RGB")
                img = img.resize(IMG_SIZE, Image.Resampling.BILINEAR)
                arr = np.asarray(img, dtype=np.float32)

            expected_shape = (IMG_SIZE[1], IMG_SIZE[0], 3)
            if arr.shape != expected_shape:
                raise ValueError(f"Unexpected image shape: {arr.shape}")

            arrays.append(arr)
            valid_paths.append(str(p))
        except (UnidentifiedImageError, OSError, ValueError) as exc:
            errors.append((str(p), repr(exc)))

    if not arrays:
        return None, [], errors

    x = preprocess_input(np.stack(arrays, axis=0))
    return x, valid_paths, errors

# ------------------------------------------------------------
# Quality Gate: two-batch forward + miniature SVM fit/predict.
# VGG16 is frozen, therefore backward is not applicable.
# ------------------------------------------------------------
smoke_df = inventory[inventory["split"] == "train"].copy()
smoke_count = min(len(smoke_df), BATCH_SIZE * 2)
if smoke_count < 4:
    raise RuntimeError("Smoke test requires at least 4 train images.")

smoke_df = smoke_df.iloc[:smoke_count].copy()
smoke_features = []
smoke_labels = []

for start in range(0, smoke_count, BATCH_SIZE):
    chunk = smoke_df.iloc[start:start + BATCH_SIZE]
    x, valid_paths, errors = load_batch(chunk["output_path"].tolist())

    if errors:
        raise RuntimeError(f"Smoke test image read failure: {errors[:3]}")
    if x is None:
        raise RuntimeError("Smoke test produced an empty batch.")

    feats = feature_extractor.predict(x, verbose=0)
    if feats.ndim != 2 or feats.shape[1] != FEATURE_DIM:
        raise RuntimeError(f"Smoke test feature shape failure: {feats.shape}")
    if not np.isfinite(feats).all():
        raise RuntimeError("Smoke test numerical gate failed: NaN/Inf feature.")

    smoke_features.append(feats.astype(np.float32))
    smoke_labels.extend(
        chunk["numeric_label"].astype(np.int64).tolist()
    )

smoke_X = np.concatenate(smoke_features, axis=0)
smoke_y = np.asarray(smoke_labels, dtype=np.int64)

if len(np.unique(smoke_y)) < 2:
    # Deterministic fallback: take a small balanced sample from train.
    balanced_parts = []
    per_class = min(4, int((inventory["split"] == "train").sum()))
    for label in [0, 1]:
        part = inventory[
            (inventory["split"] == "train")
            & (inventory["numeric_label"] == label)
        ].head(per_class)
        balanced_parts.append(part)
    smoke_df = pd.concat(balanced_parts, ignore_index=True)

    x, valid_paths, errors = load_batch(smoke_df["output_path"].tolist())
    if errors or x is None:
        raise RuntimeError("Balanced smoke test image loading failed.")
    smoke_X = feature_extractor.predict(x, verbose=0).astype(np.float32)
    smoke_y = smoke_df["numeric_label"].astype(np.int64).to_numpy()

smoke_scaler = StandardScaler()
smoke_Xs = smoke_scaler.fit_transform(smoke_X)
smoke_svm = SVC(kernel="linear", C=1.0, class_weight="balanced")
smoke_svm.fit(smoke_Xs, smoke_y)
smoke_pred = smoke_svm.predict(smoke_Xs)

if len(smoke_pred) != len(smoke_y):
    raise RuntimeError("Smoke test SVM prediction length mismatch.")

smoke_audit = {
    "status": "PASSED",
    "batches_target": 2,
    "feature_forward": "PASSED",
    "numerical_check": "PASSED",
    "mini_svm_fit_predict": "PASSED",
    "backward": "NOT_APPLICABLE",
    "reason": "VGG16 is a frozen feature extractor; no gradient-based CNN training occurs.",
}
atomic_json_dump(smoke_audit, ARTIFACTS_DIR / "smoke_test.json")

def extract_features_for_split(split_name):
    df = inventory[inventory["split"] == split_name].copy().reset_index(drop=True)
    if df.empty:
        raise RuntimeError(f"Split is empty: {split_name}")

    features_list = []
    kept_rows = []
    bad_files = []

    for start in tqdm(
        range(0, len(df), BATCH_SIZE),
        desc=f"VGG16 features — {split_name}",
    ):
        chunk = df.iloc[start:start + BATCH_SIZE]
        x, valid_paths, errors = load_batch(chunk["output_path"].tolist())
        bad_files.extend(errors)

        if x is None:
            continue

        feats = feature_extractor.predict(x, verbose=0)
        if feats.ndim != 2 or feats.shape[1] != FEATURE_DIM:
            raise RuntimeError(
                f"{split_name}: invalid feature shape {feats.shape}"
            )
        if not np.isfinite(feats).all():
            raise RuntimeError(
                f"{split_name}: numerical gate failed (NaN/Inf features)."
            )

        path_to_row = {
            str(path): row
            for path, row in zip(chunk["output_path"], chunk.to_dict("records"))
        }
        for p in valid_paths:
            kept_rows.append(path_to_row[p])

        features_list.append(feats.astype(np.float32))

    if bad_files:
        bad_df = pd.DataFrame(bad_files, columns=["path", "error"])
        atomic_dataframe_csv(
            bad_df,
            ARTIFACTS_DIR / f"{split_name}_bad_files.csv",
            index=False,
        )
        if CONFIG["quality_gates"]["fail_on_unreadable_success_image"]:
            raise RuntimeError(
                f"{split_name}: {len(bad_files)} SUCCESS images are unreadable."
            )

    if not features_list:
        raise RuntimeError(f"{split_name}: no features extracted.")

    kept = pd.DataFrame(kept_rows)
    X = np.concatenate(features_list, axis=0)
    y = kept["numeric_label"].astype(np.int64).to_numpy()

    if X.shape[0] != len(kept) or len(y) != len(kept):
        raise RuntimeError(f"{split_name}: feature/metadata count mismatch.")
    if X.shape[1] != FEATURE_DIM:
        raise RuntimeError(f"{split_name}: feature_dim mismatch.")
    if not np.isfinite(X).all():
        raise RuntimeError(f"{split_name}: NaN/Inf features.")

    # Atomic NPZ save
    target = ARTIFACTS_DIR / f"{split_name}_features.npz"
    tmp = target.with_name(target.name + ".tmp")
    with open(tmp, "wb") as f:
        np.savez_compressed(
            f,
            features=X,
            labels=y,
            sample_ids=kept["sample_id"].astype(str).to_numpy(dtype=object),
            paths=kept["output_path"].astype(str).to_numpy(dtype=object),
            source_videos=kept["source_video"].astype(str).to_numpy(dtype=object),
            sha256=kept["sha256"].astype(str).to_numpy(dtype=object),
        )
    with np.load(tmp, allow_pickle=True) as check:
        if check["features"].shape != X.shape:
            raise RuntimeError(f"NPZ validation failed: {tmp}")
    os.replace(tmp, target)

    return {
        "X": X,
        "y": y,
        "sample_ids": kept["sample_id"].astype(str).tolist(),
        "paths": kept["output_path"].astype(str).tolist(),
        "source_videos": kept["source_video"].astype(str).tolist(),
    }

train_data = extract_features_for_split("train")
val_data = extract_features_for_split("val")
test_data = extract_features_for_split("test")

X_train, y_train = train_data["X"], train_data["y"]
X_val, y_val = val_data["X"], val_data["y"]
X_test, y_test = test_data["X"], test_data["y"]

print("Train:", X_train.shape, y_train.shape)
print("Val  :", X_val.shape, y_val.shape)
print("Test :", X_test.shape, y_test.shape)


In [ ]:
# ============================================================
# 8) VALIDATION-ONLY SVM MODEL SELECTION
# ============================================================

selection_scaler = StandardScaler()
X_train_scaled = selection_scaler.fit_transform(X_train)
X_val_scaled = selection_scaler.transform(X_val)

if not np.isfinite(X_train_scaled).all():
    raise RuntimeError("Numerical gate failed after train scaling.")
if not np.isfinite(X_val_scaled).all():
    raise RuntimeError("Numerical gate failed after validation scaling.")

svm_cfg = CONFIG["svm"]
search_cfg = svm_cfg["search_space"]

candidates = [
    {"kernel": "linear", "C": float(C), "gamma": "scale"}
    for C in search_cfg["linear_C"]
]
candidates.extend(
    {
        "kernel": "rbf",
        "C": float(C),
        "gamma": gamma,
    }
    for C in search_cfg["rbf_C"]
    for gamma in search_cfg["rbf_gamma"]
)

rows = []

for params in tqdm(candidates, desc="SVM validation search"):
    candidate = SVC(
        kernel=params["kernel"],
        C=params["C"],
        gamma=params["gamma"],
        class_weight=svm_cfg["class_weight"],
        probability=False,
        random_state=CONFIG["seed"],
    )

    started = time.time()
    candidate.fit(X_train_scaled, y_train)
    fit_seconds = time.time() - started

    pred = candidate.predict(X_val_scaled)
    score = candidate.decision_function(X_val_scaled)

    if not np.isfinite(score).all():
        raise RuntimeError("Numerical gate failed: invalid SVM decision scores.")

    rows.append({
        **params,
        "val_accuracy": float(accuracy_score(y_val, pred)),
        "val_precision": float(precision_score(y_val, pred, zero_division=0)),
        "val_recall": float(recall_score(y_val, pred, zero_division=0)),
        "val_f1": float(f1_score(y_val, pred, zero_division=0)),
        "val_roc_auc": float(roc_auc_score(y_val, score)),
        "fit_seconds": float(fit_seconds),
    })

search_df = pd.DataFrame(rows)
sort_columns = list(svm_cfg["selection_sort"])
search_df = search_df.sort_values(
    sort_columns,
    ascending=[False] * len(sort_columns),
).reset_index(drop=True)

if search_df.empty:
    raise RuntimeError("Validation search produced no candidate.")

atomic_dataframe_csv(
    search_df,
    METRICS_DIR / "svm_validation_search.csv",
    index=False,
)

best_gamma = search_df.loc[0, "gamma"]
if isinstance(best_gamma, str) and best_gamma not in {"scale", "auto"}:
    best_gamma = float(best_gamma)

best_params = {
    "kernel": str(search_df.loc[0, "kernel"]),
    "C": float(search_df.loc[0, "C"]),
    "gamma": best_gamma,
}

atomic_json_dump(best_params, METRICS_DIR / "best_svm_params.json")
display(search_df)
print("Best parameters:", best_params)


In [ ]:
# ============================================================
# 9) FINAL TRAIN+VAL FIT + ATOMIC MODEL CHECKPOINT
# ============================================================

X_trainval = np.concatenate([X_train, X_val], axis=0)
y_trainval = np.concatenate([y_train, y_val], axis=0)

final_scaler = StandardScaler()
X_trainval_scaled = final_scaler.fit_transform(X_trainval)
X_test_scaled = final_scaler.transform(X_test)

if not np.isfinite(X_trainval_scaled).all():
    raise RuntimeError("Numerical gate failed after train+val scaling.")
if not np.isfinite(X_test_scaled).all():
    raise RuntimeError("Numerical gate failed after test transform.")

final_svm = SVC(
    kernel=best_params["kernel"],
    C=best_params["C"],
    gamma=best_params["gamma"],
    class_weight=CONFIG["svm"]["class_weight"],
    probability=bool(CONFIG["svm"]["probability_final"]),
    random_state=CONFIG["seed"],
)

started = time.time()
final_svm.fit(X_trainval_scaled, y_trainval)
final_fit_seconds = time.time() - started

MODEL_PATH = CHECKPOINTS_DIR / "best_model.joblib"
SCALER_PATH = CHECKPOINTS_DIR / "best_scaler.joblib"
STATE_PATH = CHECKPOINTS_DIR / "best_state.json"

atomic_joblib_dump(final_svm, MODEL_PATH)
atomic_joblib_dump(final_scaler, SCALER_PATH)

checkpoint_state = {
    "run_id": RUN_ID,
    "model_type": "sklearn.svm.SVC",
    "feature_extractor": "VGG16_frozen_imagenet",
    "best_validation_params": best_params,
    "final_fit_seconds": float(final_fit_seconds),
    "seed": int(CONFIG["seed"]),
    "train_samples": int(len(y_train)),
    "val_samples": int(len(y_val)),
    "trainval_samples": int(len(y_trainval)),
    "config_path": str(RUN_DIR / "config_resolved.yaml"),
    "note": (
        "This experiment has no epoch/optimizer/scheduler state because VGG16 is frozen "
        "and SVM training is a single deterministic fit operation."
    ),
}
atomic_json_dump(checkpoint_state, STATE_PATH)

# Inference quality gate: load from disk in a fresh object path and predict.
loaded_model = joblib.load(MODEL_PATH)
loaded_scaler = joblib.load(SCALER_PATH)

smoke_n = min(16, len(X_test))
loaded_test = loaded_scaler.transform(X_test[:smoke_n])
pred_original = final_svm.predict(X_test_scaled[:smoke_n])
pred_loaded = loaded_model.predict(loaded_test)

if not np.array_equal(pred_original, pred_loaded):
    raise RuntimeError("Inference quality gate failed after model reload.")

checkpoint_audit = {
    "atomic_model_save": "PASSED",
    "atomic_scaler_save": "PASSED",
    "fresh_reload_inference": "PASSED",
    "checkpoint_type": "joblib + state JSON",
    "epoch_state": "NOT_APPLICABLE",
    "reason": "Single-fit frozen-feature + SVM pipeline; no epoch-based optimizer state exists.",
}
atomic_json_dump(
    checkpoint_audit,
    ARTIFACTS_DIR / "checkpoint_test.json",
)

print("Final model checkpoint quality gate: PASSED")


In [ ]:
# ============================================================
# 10) FINAL TEST — TEST IS USED ONLY HERE
# ============================================================

test_pred = loaded_model.predict(X_test_scaled)
proba = loaded_model.predict_proba(X_test_scaled)

class_to_column = {
    int(cls): idx
    for idx, cls in enumerate(loaded_model.classes_)
}
if 1 not in class_to_column:
    raise RuntimeError("FAKE=1 class missing from final model classes_.")

fake_probability = proba[:, class_to_column[1]]
decision_score = loaded_model.decision_function(X_test_scaled)

if not np.isfinite(fake_probability).all():
    raise RuntimeError("Numerical gate failed: NaN/Inf test probabilities.")
if not np.isfinite(decision_score).all():
    raise RuntimeError("Numerical gate failed: NaN/Inf decision scores.")

frame_metrics = {
    "accuracy": float(accuracy_score(y_test, test_pred)),
    "precision": float(precision_score(y_test, test_pred, zero_division=0)),
    "recall": float(recall_score(y_test, test_pred, zero_division=0)),
    "f1_score": float(f1_score(y_test, test_pred, zero_division=0)),
    "roc_auc": float(roc_auc_score(y_test, fake_probability)),
    "average_precision": float(average_precision_score(y_test, fake_probability)),
    "n_test_frames": int(len(y_test)),
}

report_df = pd.DataFrame(
    classification_report(
        y_test,
        test_pred,
        labels=[0, 1],
        target_names=["REAL", "FAKE"],
        output_dict=True,
        zero_division=0,
    )
).T

cm = confusion_matrix(y_test, test_pred, labels=[0, 1])
cm_df = pd.DataFrame(
    cm,
    index=["true_REAL", "true_FAKE"],
    columns=["pred_REAL", "pred_FAKE"],
)

predictions_df = pd.DataFrame({
    "sample_id": test_data["sample_ids"],
    "source_video": test_data["source_videos"],
    "image_path": test_data["paths"],
    "true_label": y_test,
    "true_class": np.where(y_test == 1, "FAKE", "REAL"),
    "predicted_label": test_pred,
    "predicted_class": np.where(test_pred == 1, "FAKE", "REAL"),
    "fake_probability": fake_probability,
    "decision_score": decision_score,
    "run_id": RUN_ID,
})

atomic_dataframe_csv(
    report_df,
    METRICS_DIR / "classification_report.csv",
    index=True,
)
atomic_dataframe_csv(
    cm_df,
    METRICS_DIR / "confusion_matrix.csv",
    index=True,
)
atomic_dataframe_csv(
    predictions_df,
    PREDICTIONS_DIR / "test_predictions.csv",
    index=False,
)
atomic_json_dump(
    frame_metrics,
    METRICS_DIR / "frame_metrics.json",
)

print(json.dumps(frame_metrics, indent=2))
display(report_df)
display(cm_df)


In [ ]:
# ============================================================
# 11) VIDEO-LEVEL METRICS — ONLY IF SOURCE_VIDEO IS VERIFIED
# ============================================================

video_metrics = None
video_level_status = "NOT_AVAILABLE"

if source_video_gate_status == "PASSED":
    video_df = (
        predictions_df
        .groupby(["source_video", "true_label", "true_class"], as_index=False)
        .agg(
            fake_probability=("fake_probability", "mean"),
            frame_count=("sample_id", "size"),
        )
    )

    if len(video_df) < len(predictions_df) and video_df["true_label"].nunique() == 2:
        video_df["predicted_label"] = (
            video_df["fake_probability"] >= 0.5
        ).astype(int)
        video_df["predicted_class"] = np.where(
            video_df["predicted_label"] == 1,
            "FAKE",
            "REAL",
        )

        video_metrics = {
            "accuracy": float(
                accuracy_score(
                    video_df["true_label"],
                    video_df["predicted_label"],
                )
            ),
            "precision": float(
                precision_score(
                    video_df["true_label"],
                    video_df["predicted_label"],
                    zero_division=0,
                )
            ),
            "recall": float(
                recall_score(
                    video_df["true_label"],
                    video_df["predicted_label"],
                    zero_division=0,
                )
            ),
            "f1_score": float(
                f1_score(
                    video_df["true_label"],
                    video_df["predicted_label"],
                    zero_division=0,
                )
            ),
            "roc_auc": float(
                roc_auc_score(
                    video_df["true_label"],
                    video_df["fake_probability"],
                )
            ),
            "n_test_videos": int(len(video_df)),
        }
        video_level_status = "AVAILABLE"

        atomic_dataframe_csv(
            video_df,
            PREDICTIONS_DIR / "video_level_predictions.csv",
            index=False,
        )

atomic_json_dump(
    {
        "status": video_level_status,
        "metrics": video_metrics,
        "source_video_gate": source_video_gate_status,
    },
    METRICS_DIR / "video_level_metrics.json",
)

print("Video-level status:", video_level_status)


In [ ]:
# ============================================================
# 12) SSOT FIGURE QUALITY STANDARD
# ============================================================

FIGSIZE = tuple(CONFIG["figures"]["figsize"])
DPI = int(CONFIG["figures"]["dpi"])
MIN_SHORT_SIDE = int(CONFIG["figures"]["minimum_short_side_px"])

def validate_figure(path: Path):
    if not path.is_file():
        raise FileNotFoundError(f"Figure was not saved: {path}")
    with Image.open(path) as img:
        size = img.size
    if min(size) < MIN_SHORT_SIDE:
        raise RuntimeError(
            f"Figure quality gate failed: {path.name} size={size}, "
            f"minimum short side={MIN_SHORT_SIDE}px"
        )
    return size

# Confusion Matrix
fig, ax = plt.subplots(figsize=FIGSIZE, dpi=DPI)
image = ax.imshow(cm)
ax.set_title(
    "Confusion Matrix — Eye ROI VGG16 Features + SVM",
    fontsize=14,
    fontweight="bold",
    pad=12,
)
ax.set_xlabel("Predicted Label", fontsize=11)
ax.set_ylabel("True Label", fontsize=11)
ax.set_xticks([0, 1], labels=["REAL", "FAKE"])
ax.set_yticks([0, 1], labels=["REAL", "FAKE"])

for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        ax.text(j, i, str(cm[i, j]), ha="center", va="center", fontsize=11)

fig.tight_layout()
cm_path = FIGURES_DIR / f"confusion_matrix_{RUN_ID}.png"
fig.savefig(cm_path, dpi=DPI, bbox_inches="tight")
plt.close(fig)
validate_figure(cm_path)

# ROC Curve
fpr, tpr, _ = roc_curve(y_test, fake_probability)
fig, ax = plt.subplots(figsize=FIGSIZE, dpi=DPI)
ax.plot(
    fpr,
    tpr,
    linewidth=2,
    label=f"ROC-AUC = {frame_metrics['roc_auc']:.4f}",
)
ax.plot([0, 1], [0, 1], linestyle="--")
ax.set_title(
    "ROC Curve — Eye ROI VGG16 Features + SVM",
    fontsize=14,
    fontweight="bold",
    pad=12,
)
ax.set_xlabel("False Positive Rate", fontsize=11)
ax.set_ylabel("True Positive Rate", fontsize=11)
ax.legend(frameon=True, loc="lower right")
ax.grid(True, alpha=0.25)
fig.tight_layout()
roc_path = FIGURES_DIR / f"roc_curve_{RUN_ID}.png"
fig.savefig(roc_path, dpi=DPI, bbox_inches="tight")
plt.close(fig)
validate_figure(roc_path)

# Precision-Recall Curve
precision_curve, recall_curve, _ = precision_recall_curve(
    y_test,
    fake_probability,
)
fig, ax = plt.subplots(figsize=FIGSIZE, dpi=DPI)
ax.plot(
    recall_curve,
    precision_curve,
    linewidth=2,
    label=f"AP = {frame_metrics['average_precision']:.4f}",
)
ax.set_title(
    "Precision-Recall Curve — Eye ROI VGG16 Features + SVM",
    fontsize=14,
    fontweight="bold",
    pad=12,
)
ax.set_xlabel("Recall", fontsize=11)
ax.set_ylabel("Precision", fontsize=11)
ax.legend(frameon=True, loc="lower left")
ax.grid(True, alpha=0.25)
fig.tight_layout()
pr_path = FIGURES_DIR / f"precision_recall_curve_{RUN_ID}.png"
fig.savefig(pr_path, dpi=DPI, bbox_inches="tight")
plt.close(fig)
validate_figure(pr_path)

figure_audit = {
    "language": "English",
    "dpi": DPI,
    "minimum_short_side_px": MIN_SHORT_SIDE,
    "files": {
        cm_path.name: validate_figure(cm_path),
        roc_path.name: validate_figure(roc_path),
        pr_path.name: validate_figure(pr_path),
    },
}
atomic_json_dump(
    figure_audit,
    ARTIFACTS_DIR / "figure_quality_audit.json",
)

print(json.dumps(figure_audit, indent=2))


In [ ]:
# ============================================================
# 13) REPRODUCIBILITY / ENVIRONMENT / FINAL SUMMARY
# ============================================================

environment = {
    "timestamp_utc": datetime.now(timezone.utc).isoformat(),
    "python": sys.version,
    "platform": platform.platform(),
    "tensorflow": tf.__version__,
    "numpy": np.__version__,
    "pandas": pd.__version__,
    "sklearn": __import__("sklearn").__version__,
    "seed": CONFIG["seed"],
    "gpu_devices": [str(x) for x in tf.config.list_physical_devices("GPU")],
}
atomic_json_dump(environment, RUN_DIR / "environment.json")

try:
    requirements = subprocess.check_output(
        [sys.executable, "-m", "pip", "freeze"],
        text=True,
    )
except subprocess.CalledProcessError as exc:
    raise RuntimeError("pip freeze failed; reproducibility gate failed.") from exc

atomic_write_text(requirements, RUN_DIR / "requirements_lock.txt")

dataset_counts = (
    inventory.groupby(["split", "label"])
    .size()
    .rename("count")
    .reset_index()
    .to_dict(orient="records")
)

final_quality_gates = {
    **audit,
    "smoke_test": smoke_audit["status"],
    "checkpoint_test": checkpoint_audit["fresh_reload_inference"],
    "numerical_test": "PASSED",
    "inference_test": "PASSED",
    "figure_quality": "PASSED",
}

metrics_payload = {
    "run_id": RUN_ID,
    "experiment": CONFIG["experiment_name"],
    "region": CONFIG["region"],
    "roi_variant": CONFIG["roi_variant"],
    "feature_extractor": CONFIG["feature_extractor"],
    "classifier": "SVM",
    "best_svm_params": best_params,
    "frame_level": frame_metrics,
    "video_level_status": video_level_status,
    "video_level": video_metrics,
    "quality_gates": final_quality_gates,
}
atomic_json_dump(metrics_payload, METRICS_DIR / "metrics.json")

run_summary = {
    "status": "COMPLETED",
    "run_id": RUN_ID,
    "source_dataset": str(EYE_ROI_ROOT),
    "source_metadata": str(METADATA_CSV),
    "source_read_only_policy": True,
    "output_dir": str(RUN_DIR),
    "dataset_counts": dataset_counts,
    "feature_shapes": {
        "train": list(X_train.shape),
        "val": list(X_val.shape),
        "test": list(X_test.shape),
    },
    "best_svm_params": best_params,
    "final_fit_seconds": float(final_fit_seconds),
    "frame_level_metrics": frame_metrics,
    "video_level_status": video_level_status,
    "video_level_metrics": video_metrics,
    "quality_gates": final_quality_gates,
}
atomic_json_dump(run_summary, RUN_DIR / "run_summary.json")

logger.info("Final summary saved.")
print(json.dumps(run_summary, indent=2, ensure_ascii=False))


In [ ]:
# ============================================================
# 14) IMMUTABLE OUTPUT MANIFEST + FINAL DELIVERY GATE
# ============================================================

manifest_rows = []

for p in sorted(RUN_DIR.rglob("*")):
    if p.is_file() and p.name != "output_manifest.csv":
        manifest_rows.append({
            "relative_path": str(p.relative_to(RUN_DIR)),
            "size_bytes": int(p.stat().st_size),
            "sha256": sha256_file(p),
        })

manifest_df = pd.DataFrame(
    manifest_rows,
    columns=["relative_path", "size_bytes", "sha256"],
)
atomic_dataframe_csv(
    manifest_df,
    RUN_DIR / "output_manifest.csv",
    index=False,
)

# Required deliverables.
required_outputs = [
    RUN_DIR / "config_resolved.yaml",
    CHECKPOINTS_DIR / "best_model.joblib",
    CHECKPOINTS_DIR / "best_scaler.joblib",
    CHECKPOINTS_DIR / "best_state.json",
    METRICS_DIR / "metrics.json",
    METRICS_DIR / "classification_report.csv",
    METRICS_DIR / "confusion_matrix.csv",
    PREDICTIONS_DIR / "test_predictions.csv",
    ARTIFACTS_DIR / "metadata_canonical_ssot.csv",
    ARTIFACTS_DIR / "quality_gate_audit.json",
    ARTIFACTS_DIR / "smoke_test.json",
    ARTIFACTS_DIR / "checkpoint_test.json",
    ARTIFACTS_DIR / "figure_quality_audit.json",
    RUN_DIR / "environment.json",
    RUN_DIR / "requirements_lock.txt",
    RUN_DIR / "run_summary.json",
    RUN_DIR / "output_manifest.csv",
]

missing_required = [str(p) for p in required_outputs if not p.is_file()]
if missing_required:
    raise RuntimeError(
        "Final delivery gate failed. Missing outputs:\n" + "\n".join(missing_required)
    )

logger.info("All final quality gates PASSED.")
print("\n✅ SSOT v2.0 FINAL DELIVERY GATE: PASSED")
print("RUN_ID :", RUN_ID)
print("OUTPUT :", RUN_DIR)
display(manifest_df)


## SSOT v2.0 Compliance Notes

Bu notebook, standardın VGG16-feature-extractor + SVM deneyine uygulanabilen
maddelerini kod seviyesinde quality gate olarak uygular.

**Bilinçli ve açık uyarlamalar:**

- `last.ckpt / best.ckpt / epoch_N.ckpt` yapısı epoch-tabanlı PyTorch eğitimleri içindir.
  Bu deneyde CNN eğitilmez; VGG16 dondurulmuş feature extractor'dır ve SVM tek `fit`
  çağrısıyla eğitilir. Bu nedenle atomik `best_model.joblib`,
  `best_scaler.joblib` ve `best_state.json` üretilir.
- Forward/backward smoke testinin backward kısmı gradient tabanlı eğitim olmadığı için
  uygulanamaz. Bunun yerine 2-batch VGG16 forward, sayısal feature kontrolü ve mini
  SVM fit/predict testi yapılır.
- Kaynak metadata gerçek `source_video` bilgisini içermiyorsa bu bilgi uydurulmaz.
  Böyle bir durumda source-video split izolasyonu `NOT_VERIFIABLE` olarak açıkça raporlanır.
- Exact-image SHA-256 cross-split izolasyonu ise her koşulda zorunlu olarak uygulanır.


### Duplicate `sample_id` düzeltmesi

Kaynak göz metadata'sındaki `sample_id` alanının ROI satırı seviyesinde benzersiz
olduğu artık varsayılmaz. Aynı frame/sample altında birden fazla yüz veya ROI bulunabileceği
için kaynak ID `source_sample_id` olarak korunur; model manifestindeki yeni `sample_id`
ise source ID + video/frame/face/path bilgileri üzerinden deterministik SHA-256 ile üretilir.
Gerçekten aynı SUCCESS ROI kaydı iki kez bulunursa kod bunu gizlemez ve ayrı audit CSV'si
üreterek durur.
